# Trajectory Planner: Deep Dive Tutorial
## For Complete Beginners (19-Year-Old Engineer Edition)

This notebook explains **EVERY** formula, term, and concept with:
1. **What it does** - What is its job?
2. **Why it's needed** - What breaks without it?
3. **What if we remove it** - What goes wrong?
4. **Visual examples** - See it in action

## SECTION 1: Quaternions vs Yaw Angle

### The Problem
Imagine you're sitting in a car. The car can rotate in 3D space:
- **Roll**: Lean left/right (like tilting your head)
- **Pitch**: Lean forward/backward (like nodding)
- **Yaw**: Spin left/right (like turning steering wheel)

**Quaternion** = A way to store all 3 rotations in 4 numbers: `(x, y, z, w)`
**Yaw** = Just the left/right spin in 1 number: angle in radians

**Why conversion needed?** Path planning only cares about yaw (steering), not roll/pitch.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# VISUALIZATION: Show all 3 rotations
fig = plt.figure(figsize=(15, 4))

# Roll (tilt left/right)
ax1 = fig.add_subplot(131, projection='3d')
ax1.quiver(0, 0, 0, 1, 0, 0, color='r', arrow_length_ratio=0.1, linewidth=2, label='Forward')
ax1.quiver(0, 0, 0, 0, 1, 0, color='g', arrow_length_ratio=0.1, linewidth=2, label='Left')
ax1.quiver(0, 0, 0, 0, 0, 1, color='b', arrow_length_ratio=0.1, linewidth=2, label='Up')
# Rotate 45 degrees around forward axis (roll)
angle = math.radians(45)
ax1.quiver(0, 0, 0, 0, math.cos(angle), math.sin(angle), color='g', arrow_length_ratio=0.1, linewidth=2, linestyle='--', alpha=0.5)
ax1.quiver(0, 0, 0, 0, -math.sin(angle), math.cos(angle), color='b', arrow_length_ratio=0.1, linewidth=2, linestyle='--', alpha=0.5)
ax1.set_title('ROLL (Tilt Left/Right)\nRotation around forward axis', fontsize=10, fontweight='bold')
ax1.set_xlim([-1.5, 1.5])
ax1.set_ylim([-1.5, 1.5])
ax1.set_zlim([-1.5, 1.5])
ax1.set_xlabel('Forward')
ax1.set_ylabel('Left')
ax1.set_zlabel('Up')

# Pitch (tilt forward/backward)
ax2 = fig.add_subplot(132, projection='3d')
ax2.quiver(0, 0, 0, 1, 0, 0, color='r', arrow_length_ratio=0.1, linewidth=2)
ax2.quiver(0, 0, 0, 0, 1, 0, color='g', arrow_length_ratio=0.1, linewidth=2)
ax2.quiver(0, 0, 0, 0, 0, 1, color='b', arrow_length_ratio=0.1, linewidth=2)
# Rotate 45 degrees around left axis (pitch)
ax2.quiver(0, 0, 0, math.cos(angle), 0, math.sin(angle), color='r', arrow_length_ratio=0.1, linewidth=2, linestyle='--', alpha=0.5)
ax2.quiver(0, 0, 0, -math.sin(angle), 0, math.cos(angle), color='b', arrow_length_ratio=0.1, linewidth=2, linestyle='--', alpha=0.5)
ax2.set_title('PITCH (Tilt Forward/Back)\nRotation around left axis', fontsize=10, fontweight='bold')
ax2.set_xlim([-1.5, 1.5])
ax2.set_ylim([-1.5, 1.5])
ax2.set_zlim([-1.5, 1.5])
ax2.set_xlabel('Forward')
ax2.set_ylabel('Left')
ax2.set_zlabel('Up')

# Yaw (spin left/right)
ax3 = fig.add_subplot(133, projection='3d')
ax3.quiver(0, 0, 0, 1, 0, 0, color='r', arrow_length_ratio=0.1, linewidth=2)
ax3.quiver(0, 0, 0, 0, 1, 0, color='g', arrow_length_ratio=0.1, linewidth=2)
ax3.quiver(0, 0, 0, 0, 0, 1, color='b', arrow_length_ratio=0.1, linewidth=2)
# Rotate 45 degrees around up axis (yaw)
ax3.quiver(0, 0, 0, math.cos(angle), math.sin(angle), 0, color='r', arrow_length_ratio=0.1, linewidth=2, linestyle='--', alpha=0.5)
ax3.quiver(0, 0, 0, -math.sin(angle), math.cos(angle), 0, color='g', arrow_length_ratio=0.1, linewidth=2, linestyle='--', alpha=0.5)
ax3.set_title('YAW (Spin Left/Right)\nRotation around up axis', fontsize=10, fontweight='bold')
ax3.set_xlim([-1.5, 1.5])
ax3.set_ylim([-1.5, 1.5])
ax3.set_zlim([-1.5, 1.5])
ax3.set_xlabel('Forward')
ax3.set_ylabel('Left')
ax3.set_zlabel('Up')

plt.tight_layout()
plt.show()

print("\n" + "="*80)
print("WHY ONLY YAW FOR PATH PLANNING?")
print("="*80)
print("""
For a car driving on FLAT GROUND:
  - Roll = Not important (car doesn't tilt on flat road)
  - Pitch = Not important (car doesn't lean forward/back on flat road)
  - Yaw = CRITICAL! (determines which direction car is pointing)

So we extract JUST the yaw angle from the quaternion.
""")

### The Math: Quaternion to Yaw Formula

**Formula**:
```
yaw = atan2(2*(w*z + x*y), 1 - 2*(y² + z²))
```

**Breaking it down:**
- `w, x, y, z` = Quaternion components
- `2*(w*z + x*y)` = Numerator (sine-like term)
- `1 - 2*(y² + z²)` = Denominator (cosine-like term)
- `atan2` = "angle to tangent" function

**Why atan2 instead of atan?**
- `atan` only gives -90° to +90°
- `atan2` gives full -180° to +180° (all directions)

In [ ]:
def quaternion_to_yaw(q_w, q_x, q_y, q_z):
    """
    Convert quaternion (w, x, y, z) to yaw angle.
    
    STEP 1: Calculate numerator (for sine)
    numerator = 2 * (w*z + x*y)
    
    STEP 2: Calculate denominator (for cosine)  
    denominator = 1 - 2 * (y² + z²)
    
    STEP 3: Use atan2 to get angle
    yaw = atan2(numerator, denominator)
    
    OUTPUT: yaw in radians [-π, π]
    """
    numerator = 2 * (q_w * q_z + q_x * q_y)
    denominator = 1 - 2 * (q_y**2 + q_z**2)
    yaw = math.atan2(numerator, denominator)
    return yaw

# TEST: Convert different quaternions
test_cases = [
    (1, 0, 0, 0, "0° (pointing forward)"),
    (0.707, 0, 0, 0.707, "90° (pointing left)"),
    (0, 0, 0, 1, "180° (pointing backward)"),
    (0.707, 0, 0, -0.707, "-90° (pointing right)"),
]

print("QUATERNION TO YAW CONVERSION EXAMPLES:")
print("="*70)
for w, x, y, z, description in test_cases:
    yaw_rad = quaternion_to_yaw(w, x, y, z)
    yaw_deg = math.degrees(yaw_rad)
    print(f"\nQuaternion: ({w:.3f}, {x:.3f}, {y:.3f}, {z:.3f})")
    print(f"  → Yaw: {yaw_rad:.4f} radians = {yaw_deg:.1f}°")
    print(f"  → Meaning: {description}")

# VISUALIZATION: Show how quaternion rotates the car
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle('Car Orientation for Different Quaternions', fontsize=14, fontweight='bold')

for idx, (w, x, y, z, desc) in enumerate(test_cases):
    ax = axes[idx // 2, idx % 2]
    
    yaw = quaternion_to_yaw(w, x, y, z)
    
    # Draw car
    car_x = [0, 0.5 * math.cos(yaw)]
    car_y = [0, 0.5 * math.sin(yaw)]
    ax.arrow(0, 0, car_x[1]*0.9, car_y[1]*0.9, head_width=0.15, head_length=0.1, fc='blue', ec='blue', linewidth=2)
    ax.plot(0, 0, 'ko', markersize=8)
    
    # Draw direction labels
    ax.text(0.6, 0, 'East', fontsize=10)
    ax.text(0, 0.6, 'North', fontsize=10)
    ax.text(-0.8, 0, 'West', fontsize=10)
    ax.text(0, -0.8, 'South', fontsize=10)
    
    # Draw compass circle
    circle = plt.Circle((0, 0), 0.5, fill=False, linestyle='--', color='gray', alpha=0.3)
    ax.add_patch(circle)
    
    ax.set_xlim(-1, 1)
    ax.set_ylim(-1, 1)
    ax.set_aspect('equal')
    ax.set_title(f"{desc}\n(Quat: {w:.2f}, {x:.2f}, {y:.2f}, {z:.2f})")
    ax.grid(True, alpha=0.3)
    ax.axhline(0, color='k', linewidth=0.5)
    ax.axvline(0, color='k', linewidth=0.5)

plt.tight_layout()
plt.show()

### What If We Remove This Conversion?

❌ **Without quaternion_to_yaw:**
- We'd have 4 numbers (w, x, y, z) but need 1 angle
- All coordinate transformations would break
- Can't calculate "is target to my left or right?"
- Path following would be impossible

✅ **With conversion:**
- Simple angle: -180° to +180°
- Easy trigonometry: cos(yaw), sin(yaw)
- Path planning works correctly

---

## SECTION 2: Ego-Frame Coordinate Transformation

### The Problem

Imagine you're in a car:
- **World Frame**: Maps use coordinates like (123m North, 456m East)
- **Your Frame**: You think "the store is 100m ahead and 20m to my left"

We need to convert world coordinates → your perspective

In [ ]:
# VISUAL EXPLANATION
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# WORLD FRAME
ax1.set_title('WORLD FRAME (Map Perspective)', fontsize=12, fontweight='bold')
ax1.set_xlabel('X (East) [meters]')
ax1.set_ylabel('Y (North) [meters]')

# Draw car at position (2, 1) pointing at angle 30°
car_x, car_y = 2, 1
car_yaw = math.radians(30)  # 30 degrees

# Car heading arrow
arrow_len = 1
ax1.arrow(car_x, car_y, arrow_len*math.cos(car_yaw), arrow_len*math.sin(car_yaw), 
         head_width=0.2, head_length=0.15, fc='blue', ec='blue', linewidth=2, label='Car heading')
ax1.plot(car_x, car_y, 'bo', markersize=12)  # Car center

# Draw waypoint
waypoint_x, waypoint_y = 4, 3
ax1.plot(waypoint_x, waypoint_y, 'r*', markersize=20, label='Target waypoint')

# Draw vector from car to waypoint
ax1.arrow(car_x, car_y, (waypoint_x-car_x)*0.9, (waypoint_y-car_y)*0.9, 
         head_width=0.15, head_length=0.1, fc='red', ec='red', linestyle='--', alpha=0.5)

# Draw grid
ax1.grid(True, alpha=0.3)
ax1.set_xlim(0, 5)
ax1.set_ylim(0, 4)
ax1.legend()
ax1.text(car_x-0.3, car_y-0.4, 'Car', fontsize=10)
ax1.text(waypoint_x+0.1, waypoint_y+0.2, 'Target', fontsize=10)

# EGO FRAME (Car's perspective)
ax2.set_title("EGO FRAME (Car's Perspective)", fontsize=12, fontweight='bold')
ax2.set_xlabel('Forward/Backward (m)')
ax2.set_ylabel('Left/Right (m)')

# In ego frame, car is always at origin looking forward
ax2.arrow(0, 0, 1.5, 0, head_width=0.2, head_length=0.15, fc='blue', ec='blue', linewidth=2, label='Forward (what car sees)')
ax2.plot(0, 0, 'bo', markersize=12, label='Car (always here in ego frame)')

# Transform waypoint to ego frame
# Formula: x_ego = cos(-yaw) * dx - sin(-yaw) * dy
#          y_ego = sin(-yaw) * dx + cos(-yaw) * dy
dx = waypoint_x - car_x
dy = waypoint_y - car_y

x_ego = math.cos(-car_yaw) * dx - math.sin(-car_yaw) * dy
y_ego = math.sin(-car_yaw) * dx + math.cos(-car_yaw) * dy

ax2.plot(x_ego, y_ego, 'r*', markersize=20, label='Target (in ego frame)')
ax2.arrow(0, 0, x_ego*0.9, y_ego*0.9, head_width=0.15, head_length=0.1, fc='red', ec='red', linestyle='--', alpha=0.5)

ax2.grid(True, alpha=0.3)
ax2.set_xlim(-0.5, 3)
ax2.set_ylim(-1.5, 1.5)
ax2.axhline(0, color='k', linewidth=1)
ax2.axvline(0, color='k', linewidth=1)
ax2.legend()

plt.tight_layout()
plt.show()

print(f"\nWORLD FRAME COORDINATES:")
print(f"  Car position: ({car_x:.1f}, {car_y:.1f})")
print(f"  Car heading: {math.degrees(car_yaw):.1f}°")
print(f"  Target: ({waypoint_x:.1f}, {waypoint_y:.1f})")
print(f"  Vector from car to target: ({dx:.1f}, {dy:.1f})")
print(f"\nEGO FRAME COORDINATES (Car's perspective):")
print(f"  Target is: {x_ego:.2f}m AHEAD and {y_ego:.2f}m to the {'LEFT' if y_ego > 0 else 'RIGHT'}")

### The Formula Explained

**Formula:**
```
x_ego = cos(-yaw) * (waypoint_x - car_x) - sin(-yaw) * (waypoint_y - car_y)
y_ego = sin(-yaw) * (waypoint_x - car_x) + cos(-yaw) * (waypoint_y - car_y)
```

**Breaking it down:**

1. **Calculate difference vector:**
   - `dx = waypoint_x - car_x`  (how far east is target?)
   - `dy = waypoint_y - car_y`  (how far north is target?)

2. **Apply rotation matrix:**
   - This is a 2D rotation by `-yaw` angle
   - Rotates the difference vector into car's frame

3. **Result:**
   - `x_ego` = distance ahead (+) or behind (-)
   - `y_ego` = distance left (+) or right (-)

**Why -yaw?** Because we're UN-rotating the world to match the car's orientation.

In [ ]:
def world_to_ego(waypoint_x, waypoint_y, car_x, car_y, car_yaw):
    """
    Transform world coordinates to ego (car's) frame.
    
    INPUT: Waypoint in world frame, car position and heading
    OUTPUT: Where is waypoint relative to car?
    """
    # Step 1: Vector from car to waypoint
    dx = waypoint_x - car_x
    dy = waypoint_y - car_y
    
    # Step 2: Rotate by -yaw to go into car's frame
    # Using 2D rotation matrix:
    # [cos(θ)  -sin(θ)] [dx]
    # [sin(θ)   cos(θ)] [dy]
    x_ego = math.cos(-car_yaw) * dx - math.sin(-car_yaw) * dy
    y_ego = math.sin(-car_yaw) * dx + math.cos(-car_yaw) * dy
    
    return x_ego, y_ego

# TEST: What if car is pointing different directions?
print("TEST: Same waypoint (4, 3), car at (2, 1), but different headings")
print("="*70)

headings = [
    (0, "0° (East)"),
    (math.pi/2, "90° (North)"),
    (math.pi, "180° (West)"),
    (-math.pi/2, "-90° (South)"),
    (math.pi/4, "45° (Northeast)"),
]

for heading, description in headings:
    x_ego, y_ego = world_to_ego(4, 3, 2, 1, heading)
    distance = math.sqrt(x_ego**2 + y_ego**2)
    direction = "LEFT" if y_ego > 0 else "RIGHT"
    print(f"\nCar heading {description}:")
    print(f"  Target is {x_ego:6.2f}m ahead and {y_ego:6.2f}m to the {direction}")
    print(f"  Total distance: {distance:.2f}m")

### The Rotation Matrix (Deep Dive)

The math behind the transformation is a **2D rotation matrix**:

In [ ]:
# VISUAL: Show how rotation matrix works
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Original vector
ax = axes[0]
ax.set_title('Step 1: Original Vector (World Frame)', fontweight='bold')
ax.arrow(0, 0, 2, 2, head_width=0.2, head_length=0.15, fc='blue', ec='blue', linewidth=2)
ax.text(2.2, 2.2, 'Target\n(dx=2, dy=2)', fontsize=11, fontweight='bold')
ax.set_xlim(-1, 4)
ax.set_ylim(-1, 4)
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')
ax.axhline(0, color='k', linewidth=0.5)
ax.axvline(0, color='k', linewidth=0.5)
ax.set_xlabel('X (East)')
ax.set_ylabel('Y (North)')

# Car heading
ax = axes[1]
ax.set_title('Step 2: Car Heading (Rotation Angle)', fontweight='bold')
car_yaw = math.pi / 4  # 45 degrees
ax.arrow(0, 0, 1, 0, head_width=0.15, head_length=0.1, fc='green', ec='green', linewidth=2, label='Car forward')
ax.plot([0, math.cos(car_yaw)], [0, math.sin(car_yaw)], 'g--', linewidth=2, alpha=0.5)
ax.text(math.cos(car_yaw)+0.2, math.sin(car_yaw)+0.2, f'θ = {math.degrees(car_yaw):.0f}°', fontsize=11, fontweight='bold')
ax.set_xlim(-1, 2)
ax.set_ylim(-1, 2)
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')
ax.axhline(0, color='k', linewidth=0.5)
ax.axvline(0, color='k', linewidth=0.5)
ax.set_xlabel('X (East)')
ax.set_ylabel('Y (North)')

# Rotated vector (ego frame)
ax = axes[2]
ax.set_title('Step 3: Vector in Ego Frame (Rotated)', fontweight='bold')

# Apply rotation
dx, dy = 2, 2
x_ego = math.cos(-car_yaw) * dx - math.sin(-car_yaw) * dy
y_ego = math.sin(-car_yaw) * dx + math.cos(-car_yaw) * dy

ax.arrow(0, 0, 1.5, 0, head_width=0.15, head_length=0.1, fc='green', ec='green', linewidth=2, alpha=0.5, label='Forward')
ax.arrow(0, 0, x_ego, y_ego, head_width=0.2, head_length=0.15, fc='red', ec='red', linewidth=2)
ax.text(x_ego+0.2, y_ego-0.3, f'Ego Vector\n(x={x_ego:.2f}, y={y_ego:.2f})', fontsize=11, fontweight='bold')
ax.set_xlim(-1, 4)
ax.set_ylim(-2, 2)
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')
ax.axhline(0, color='k', linewidth=0.5)
ax.axvline(0, color='k', linewidth=0.5)
ax.set_xlabel('Forward/Backward')
ax.set_ylabel('Left/Right')

plt.tight_layout()
plt.show()

print(f"\nROTATION MATRIX CALCULATION:")
print(f"="*70)
print(f"\nOriginal vector (world): (dx={dx}, dy={dy})")
print(f"Car yaw angle: {math.degrees(car_yaw):.0f}°")
print(f"\nRotation matrix for angle -θ:")
print(f"┌                                    ┐")
print(f"│  cos(-θ)    -sin(-θ)  │ dx │     │")
print(f"│  sin(-θ)     cos(-θ)  │ dy │     │")
print(f"└                                    ┘")
print(f"\nSubstitute values:")
print(f"  cos({-math.degrees(car_yaw):.0f}°) = {math.cos(-car_yaw):.4f}")
print(f"  sin({-math.degrees(car_yaw):.0f}°) = {math.sin(-car_yaw):.4f}")
print(f"\nResult:")
print(f"  x_ego = {math.cos(-car_yaw):.4f} * {dx} - {math.sin(-car_yaw):.4f} * {dy} = {x_ego:.4f}")
print(f"  y_ego = {math.sin(-car_yaw):.4f} * {dx} + {math.cos(-car_yaw):.4f} * {dy} = {y_ego:.4f}")
print(f"\nInterpretation: Target is {x_ego:.2f}m ahead and {y_ego:.2f}m to the {'left' if y_ego > 0 else 'right'}")

### What If We Remove This Transformation?

❌ **Without ego-frame transform:**
```python
# WRONG: Using world coordinates directly
if target_x > car_x:  # Is target to the east?
    steer_right()  # ❌ But car is facing north!
```
- Steering would be in WRONG direction
- Car zigzags randomly
- Can't follow any path

✅ **With transformation:**
```python
# CORRECT: Using ego frame
if target_ego_y > 0:  # Is target to the left?
    steer_left()  # ✓ Correct relative to car's heading
```
- Car steers in the RIGHT direction
- Follows path smoothly

---

## SECTION 3: Pure Pursuit Algorithm

### The Concept (Simple Analogy)

**Imagine:** You're walking on a rope path in the dark.
- You hold a flashlight that only lights up 1 meter ahead
- You always walk toward the lit point
- Result: You stay on the path!

**Pure Pursuit = Same idea:**
- Car looks ahead by "lookahead distance"
- Car steers to reach that point
- Result: Car stays on path!

In [ ]:
# VISUALIZATION: Pure Pursuit in action
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Pure Pursuit Algorithm: Step by Step', fontsize=14, fontweight='bold')

def draw_pursuit_step(ax, car_pos, car_yaw, path, lookahead_dist, title):
    """Draw one step of pure pursuit"""
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('X (meters)')
    ax.set_ylabel('Y (meters)')
    
    # Draw path
    path_x = [p[0] for p in path]
    path_y = [p[1] for p in path]
    ax.plot(path_x, path_y, 'k--', linewidth=2, alpha=0.3, label='Path')
    ax.plot(path_x, path_y, 'ko', markersize=4)
    
    # Draw car
    car_x, car_y = car_pos
    car_forward_x = car_x + 0.3 * math.cos(car_yaw)
    car_forward_y = car_y + 0.3 * math.sin(car_yaw)
    ax.arrow(car_x, car_y, car_forward_x-car_x, car_forward_y-car_y,
            head_width=0.1, head_length=0.08, fc='blue', ec='blue', linewidth=2)
    ax.plot(car_x, car_y, 'bs', markersize=12, label='Car')
    
    # Find lookahead point
    min_dist = float('inf')
    lookahead_pt = path[0]
    for p in path:
        d = math.sqrt((p[0]-car_x)**2 + (p[1]-car_y)**2)
        if abs(d - lookahead_dist) < min_dist:
            min_dist = abs(d - lookahead_dist)
            lookahead_pt = p
    
    # Draw lookahead distance circle
    circle = plt.Circle((car_x, car_y), lookahead_dist, fill=False, linestyle=':', color='green', linewidth=2, alpha=0.5)
    ax.add_patch(circle)
    
    # Draw lookahead point and target
    ax.plot(lookahead_pt[0], lookahead_pt[1], 'r*', markersize=20, label='Lookahead point')
    ax.arrow(car_x, car_y, lookahead_pt[0]-car_x, lookahead_pt[1]-car_y,
            linestyle='-', color='red', alpha=0.3, linewidth=1.5)
    
    ax.set_xlim(-1, 6)
    ax.set_ylim(-1, 4)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    ax.legend(loc='upper right')
    
    return lookahead_pt

# Generate a curved path
path = [(i*0.5, 0.3*math.sin(i*0.3)) for i in range(12)]
lookahead_distance = 1.0

# Step 1: Car at start, looks ahead
ax1 = axes[0, 0]
pt1 = draw_pursuit_step(ax1, (0.5, 0.1), math.radians(0), path, lookahead_distance, 
                        "Step 1: Look ahead 1.0m")

# Step 2: Car steers toward lookahead point
ax2 = axes[0, 1]
pt2 = draw_pursuit_step(ax2, (1.0, 0.15), math.radians(15), path, lookahead_distance,
                       "Step 2: Steer to reach it")

# Step 3: Car moves forward
ax3 = axes[1, 0]
pt3 = draw_pursuit_step(ax3, (1.8, 0.3), math.radians(25), path, lookahead_distance,
                       "Step 3: Move forward")

# Step 4: Repeat - new lookahead point
ax4 = axes[1, 1]
pt4 = draw_pursuit_step(ax4, (2.8, 0.4), math.radians(15), path, lookahead_distance,
                       "Step 4: Find new target, repeat")

plt.tight_layout()
plt.show()

print("""
PURE PURSUIT ALGORITHM FLOW:
============================

Repeat every 20ms:
  1. Find all waypoints on the path
  2. Calculate distance from car to each waypoint
  3. Find the waypoint that is exactly 'lookahead_distance' away
  4. Calculate steering angle to pass through that point
  5. Move car forward
  6. Go to step 1
""")

### The Steering Angle Formula

**Goal**: How much should we turn the wheel?

**Formula**:
```
steering_angle = atan(wheelbase * curvature)
where: curvature = (2 * y_ego) / (lookahead_distance²)
```

In [ ]:
# STEP-BY-STEP: Calculate steering angle
print("PURE PURSUIT STEERING CALCULATION")
print("="*70)

# Example values
car_x, car_y = 0, 0
car_yaw = 0  # Pointing east
target_x, target_y = 3, 0.5
wheelbase = 0.5  # Distance between front/rear axles
lookahead_dist = 1.0

print(f"\nGiven:")
print(f"  Car position: ({car_x}, {car_y})")
print(f"  Car heading: {math.degrees(car_yaw):.0f}°")
print(f"  Target point: ({target_x}, {target_y})")
print(f"  Lookahead distance: {lookahead_dist}m")
print(f"  Wheelbase: {wheelbase}m")

print(f"\n--- STEP 1: Transform to ego frame ---")
dx = target_x - car_x
dy = target_y - car_y
print(f"  Vector to target: ({dx}, {dy})")

x_ego = math.cos(-car_yaw) * dx - math.sin(-car_yaw) * dy
y_ego = math.sin(-car_yaw) * dx + math.cos(-car_yaw) * dy
print(f"  Ego frame: x_ego={x_ego:.3f}, y_ego={y_ego:.3f}")

print(f"\n--- STEP 2: Calculate curvature ---")
print(f"  Formula: curvature = (2 * y_ego) / (distance²)")
curvature = (2 * y_ego) / (x_ego ** 2)
print(f"  curvature = (2 * {y_ego:.3f}) / ({x_ego:.3f}²)")
print(f"  curvature = {curvature:.4f}")

print(f"\n--- STEP 3: Convert to steering angle ---")
print(f"  Formula: steering_angle = atan(wheelbase * curvature)")
steering_rad = math.atan(wheelbase * curvature)
steering_deg = math.degrees(steering_rad)
print(f"  steering_angle = atan({wheelbase} * {curvature:.4f})")
print(f"  steering_angle = {steering_rad:.4f} radians")
print(f"  steering_angle = {steering_deg:.2f} degrees")

print(f"\n--- RESULT ---")
print(f"  Turn steering wheel {abs(steering_deg):.2f}° to the {'LEFT' if steering_deg > 0 else 'RIGHT'}")

# VISUALIZATION: Show how steering angle depends on y_ego
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Different y_ego values
y_egos = np.linspace(-1, 1, 100)
x_ego_fixed = 3.0
steeringangles = [math.degrees(math.atan(wheelbase * (2 * y / (x_ego_fixed**2)))) for y in y_egos]

ax1.plot(y_egos, steeringangles, 'b-', linewidth=2)
ax1.axhline(0, color='k', linewidth=0.5)
ax1.axvline(0, color='k', linewidth=0.5)
ax1.grid(True, alpha=0.3)
ax1.set_xlabel('y_ego (left/right offset) [m]', fontsize=11)
ax1.set_ylabel('Steering angle [degrees]', fontsize=11)
ax1.set_title('How Much to Steer vs. Lateral Offset', fontweight='bold')
ax1.fill_between(y_egos, steeringangles, alpha=0.2)

# Plot 2: Different x_ego values (lookahead distance)
x_egos = np.linspace(0.5, 5, 100)
y_ego_fixed = 0.5
steeringangles2 = [math.degrees(math.atan(wheelbase * (2 * y_ego_fixed / (x**2)))) for x in x_egos]

ax2.plot(x_egos, steeringangles2, 'r-', linewidth=2)
ax2.grid(True, alpha=0.3)
ax2.set_xlabel('x_ego (distance ahead) [m]', fontsize=11)
ax2.set_ylabel('Steering angle [degrees]', fontsize=11)
ax2.set_title('How Much to Steer vs. Lookahead Distance', fontweight='bold')
ax2.fill_between(x_egos, steeringangles2, alpha=0.2)

plt.tight_layout()
plt.show()

### What If We Remove Pure Pursuit?

❌ **Without Pure Pursuit:**
- No path following algorithm
- Car doesn't know how to steer
- Vehicle crashes or drives randomly

✅ **With Pure Pursuit:**
- Car smoothly follows curved paths
- Adapts to different path shapes
- Stable and predictable steering

---

## SECTION 4: Dynamic Window Approach (DWA) - The Smart Overtaking

### The Concept

**Problem**: A single algorithm (Pure Pursuit) doesn't handle obstacles.

**Solution**: Try MANY options, score each, pick the best.

In [ ]:
# VISUALIZATION: DWA samples many trajectories
fig, ax = plt.subplots(figsize=(12, 8))

# Draw obstacle
obstacle_x, obstacle_y = 2, 0
circ_obstacle = plt.Circle((obstacle_x, obstacle_y), 0.3, color='red', alpha=0.7, label='Obstacle')
ax.add_patch(circ_obstacle)

# Draw car at origin
car_x, car_y, car_yaw = 0, 0, 0
ax.plot(car_x, car_y, 'bs', markersize=15, label='Car')
ax.arrow(car_x, car_y, 0.3, 0, head_width=0.1, head_length=0.08, fc='blue', ec='blue')

# Draw target
target_x, target_y = 4, 0
ax.plot(target_x, target_y, 'g*', markersize=25, label='Target')

# Sample velocities and draw trajectories
velocities = np.linspace(0.1, 0.4, 5)
angular_vels = np.linspace(-0.5, 0.5, 5)

for v in velocities:
    for w in angular_vels:
        # Simulate trajectory
        x, y, yaw = car_x, car_y, car_yaw
        traj_x, traj_y = [x], [y]
        
        for t in np.linspace(0, 1.5, 15):
            if abs(w) < 1e-3:
                x += v * math.cos(yaw) * 0.1
                y += v * math.sin(yaw) * 0.1
            else:
                x += (v/w) * (math.sin(yaw + w*0.1) - math.sin(yaw))
                y += (v/w) * (-math.cos(yaw + w*0.1) + math.cos(yaw))
                yaw += w * 0.1
            
            # Check collision
            dist_to_obstacle = math.sqrt((x - obstacle_x)**2 + (y - obstacle_y)**2)
            
            traj_x.append(x)
            traj_y.append(y)
        
        # Color based on collision
        if dist_to_obstacle > 0.5:
            color = 'green'
            alpha = 0.3
        else:
            color = 'red'
            alpha = 0.1
        
        ax.plot(traj_x, traj_y, color=color, alpha=alpha, linewidth=1)

ax.set_xlim(-1, 5)
ax.set_ylim(-2, 2)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.set_xlabel('X (meters)', fontsize=11)
ax.set_ylabel('Y (meters)', fontsize=11)
ax.set_title('DWA: Sample 25 Trajectories (5 speeds × 5 steering angles)', fontweight='bold', fontsize=12)
ax.legend(fontsize=11)

# Add annotation
ax.text(0.5, -1.5, 'RED = Collision risk (too close)\nGREEN = Safe (far enough)', 
        fontsize=10, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

print("""
DWA ALGORITHM:
==============

1. SAMPLE: Try 8×8 = 64 different (velocity, steering) combinations
   - Velocities: 0.1 to 0.4 m/s
   - Angular velocities: -30° to +30° per second

2. SIMULATE: For each combination, predict 1.5 seconds into the future
   - Use car dynamics to predict path
   - Check if it hits obstacle

3. SCORE: Calculate cost for each trajectory
   - Distance to target (30% weight)
   - Distance to obstacle (40% weight - most important!)
   - Smoothness/speed (30% weight)

4. PICK: Select the trajectory with LOWEST cost
   - Execute that steering + speed

5. REPEAT: In next 20ms, do it again
""")

### What If We Remove DWA?

❌ **Without DWA:**
- Pure Pursuit would try to go STRAIGHT through obstacle
- Car would crash
- No overtaking capability

✅ **With DWA:**
- Car evaluates 64 options
- Picks the safe one that goes around obstacle
- Smoothly overtakes

---

## SECTION 5: Steering Smoothing (AC10)

### The Problem

If steering changes too suddenly, car jerks violently. 

**Example of jerky steering:**
```
Time 0ms:   steering = 0°
Time 20ms:  steering = 20°  ← SUDDEN JUMP!
Time 40ms:  steering = 0°  ← SUDDEN JUMP!
```

The car's mechanical steering can't respond that fast, causing instability.

In [ ]:
def steering_smoothing(target_steering, previous_steering, smoothing_factor):
    """
    Smooth the steering angle to avoid jerky changes.
    
    Formula: new_steering = (1 - k) * previous + k * target
    
    where k = smoothing_factor (0 to 1)
    - k=0 → ignore target (no change)
    - k=1 → immediate change to target
    - k=0.2 → gradually change (smooth)
    """
    new_steering = (1 - smoothing_factor) * previous_steering + smoothing_factor * target_steering
    return new_steering

# Simulate steering over time
time_steps = 20
target_sequence = []

# Build a sequence of target steering angles (what we want)
for t in range(time_steps):
    if t < 5:
        target_sequence.append(0)
    elif t < 10:
        target_sequence.append(25)  # Want to turn 25 degrees
    elif t < 15:
        target_sequence.append(-15)  # Want to turn -15 degrees
    else:
        target_sequence.append(0)

# Calculate smoothed steering with different k values
smoothing_factors = [0.2, 0.5, 1.0]  # 0.2=smooth, 1.0=jerky
smoothed_results = {k: [] for k in smoothing_factors}

for k in smoothing_factors:
    smoothed = []
    previous = 0
    for target in target_sequence:
        new_steer = steering_smoothing(target, previous, k)
        smoothed.append(new_steer)
        previous = new_steer
    smoothed_results[k] = smoothed

# Plot comparison
fig, ax = plt.subplots(figsize=(12, 6))

colors = ['red', 'orange', 'green']
for k, color in zip(smoothing_factors, colors):
    label = f'k={k} (Jerky)' if k==1.0 else (f'k={k} (Smooth)' if k==0.2 else f'k={k}')
    ax.plot(range(time_steps), smoothed_results[k], 'o-', linewidth=2, markersize=6, 
           label=label, color=color, alpha=0.8)

ax.plot(range(time_steps), target_sequence, 'k--', linewidth=2, label='Target (what we want)', alpha=0.5)
ax.fill_between(range(time_steps), target_sequence, alpha=0.1, color='black')

ax.set_xlabel('Time Step (× 20ms)', fontsize=11)
ax.set_ylabel('Steering Angle (degrees)', fontsize=11)
ax.set_title('Steering Smoothing: Different Smoothing Factors', fontweight='bold', fontsize=12)
ax.grid(True, alpha=0.3)
ax.legend(fontsize=11, loc='upper left')
ax.axhline(0, color='k', linewidth=0.5, alpha=0.3)

plt.tight_layout()
plt.show()

print("STEERING SMOOTHING COMPARISON")
print("="*70)
print(f"\nSmoothing Factor k = 0.2 (AC10 default):")
print(f"  Step 5: target 0° → actual {smoothed_results[0.2][5]:.2f}°  (5% change)")
print(f"  Step 6: target 25° → actual {smoothed_results[0.2][6]:.2f}°  (gradual)")
print(f"  Step 7: target 25° → actual {smoothed_results[0.2][7]:.2f}°  (still building)")
print(f"  → Takes ~8-10 steps to reach target (smooth acceleration)")

print(f"\nSmoothing Factor k = 1.0 (no smoothing):")
print(f"  Step 5: target 0° → actual {smoothed_results[1.0][5]:.2f}°  (immediate!)")
print(f"  Step 6: target 25° → actual {smoothed_results[1.0][6]:.2f}°  (jerky jump!)")
print(f"  → Changes instantly (jerky, dangerous)")

### The Formula Explained

**Formula**:
```
smoothed_steering = (1 - k) × previous_steering + k × target_steering
```

**Where:**
- `k` = smoothing factor (0.2 by default)
- `previous_steering` = steering angle from last update
- `target_steering` = what we want to turn to
- `smoothed_steering` = what we actually command

**Why this formula?**
- Weighted average: 80% old value, 20% new value
- Changes happen gradually
- Prevents mechanical shocks

### What If We Remove Smoothing?

❌ **Without smoothing (k=1.0):**
- Steering jumps instantly from 0° to 30°
- Car lurches violently
- Mechanical wear on steering motor
- Passengers feel nauseous

✅ **With smoothing (k=0.2):**
- Steering increases gradually: 0° → 6° → 11° → 16° → ...
- Smooth turning (like human driver)
- Less mechanical stress
- Comfortable ride

---

## Summary: Complete Checklist

### Every Component and Why It Matters

In [ ]:
import pandas as pd

components = {
    'Component': [
        'Quaternion → Yaw',
        'Ego-Frame Transform',
        'Pure Pursuit',
        'DWA Algorithm',
        'Steering Smoothing',
        'Path Pruning',
        'Obstacle Detection',
        'Speed Adaptation',
        'Marker Visualization',
        'State Machine'
    ],
    'What It Does': [
        'Converts 3D orientation to steering angle',
        'Converts world coordinates to car perspective',
        'Looks ahead and steers to path waypoint',
        'Samples many options to avoid obstacles',
        'Prevents jerky steering changes',
        'Removes old waypoints for efficiency',
        'Stops if obstacle or invalid state',
        'Slows down on sharp turns',
        'Shows car position in RViz',
        'Tracks overtaking phases'
    ],
    'Why Needed': [
        'Need angle for steering calculations',
        'Must know "left/right" relative to car',
        'Main algorithm to follow paths',
        'Essential for safe overtaking',
        'AC10 requirement for smooth control',
        'Reduces computation (optimization)',
        'Safety requirement (AC8, AC9)',
        'Prevents rolling over on curves',
        'Debugging and monitoring',
        'AC2-AC5 requirements for overtaking'
    ],
    'If Removed': [
        '❌ Can\'t calculate steering angle',
        '❌ Steering in wrong direction',
        '❌ Car drives randomly',
        '❌ Crashes into obstacles',
        '❌ Jerky, unsafe vehicle behavior',
        '❌ Slower but still works',
        '❌ Vehicle doesn\'t stop → crash',
        '❌ Car rolls over, loses control',
        '❌ Can\'t see what\'s happening',
        '❌ Overtaking fails, car stuck'
    ]
}

df = pd.DataFrame(components)

# Display with better formatting
print("\n" + "="*120)
print("COMPLETE COMPONENT CHECKLIST")
print("="*120)

for idx, row in df.iterrows():
    print(f"\n{idx+1}. {row['Component']}")
    print(f"   What:    {row['What It Does']}")
    print(f"   Why:     {row['Why Needed']}")
    print(f"   Danger:  {row['If Removed']}")
    print("-" * 120)

print(f"\nTOTAL COMPONENTS: {len(df)} (all critical)")
print("\nCRITICAL (can't remove):")
for i in [0, 1, 2, 3, 4, 7, 9]:
    print(f"  • {df.loc[i, 'Component']}")
print("\nOPTIMIZATIONS (can remove, but slower):")
for i in [5, 6, 8]:
    print(f"  • {df.loc[i, 'Component']}")

---

## Final Notes for New Developers

### Key Takeaways

1. **Every line of code has a reason** - None of it is arbitrary
2. **Remove one piece = breaks the whole system**
3. **The formulas come from physics and control theory**
4. **Testing is essential** - Simulation before real vehicles
5. **Tuning matters** - Same algorithm, different parameters = totally different behavior

### Next Steps

1. Read the main `tp_planner.py` code with this notebook as reference
2. Try modifying parameters:
   - `lookahead_distance` → changes steering sensitivity
   - `steering_smoothing_factor` → changes jerkiness
   - `max_speed` → changes overall speed
3. Simulate in a ROS2 environment
4. Test with real sensor data

### Common Mistakes

❌ Tuning steering angle directly → Use Pure Pursuit formula instead
❌ Not smoothing steering → Vehicle jerks dangerously
❌ Ignoring the wheelbase → Wrong steering angles
❌ Setting lookahead_distance too small → Jerky path following
❌ Setting lookahead_distance too large → Overshoots curves

---

**Remember**: This code controls a real vehicle. Every decision affects safety. Always test thoroughly! 🚗